In [1]:
import anndata as ad
import seaborn as sns
import numpy as np
import scanpy as sc
import pandas as pd
import gc
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
mg_ad_path = '/data/scRNA/RNA/scRNA/Hammond/preprocessing/GSE121654_T_P100_LC_common_concat_zero_filter_MTremoved.h5ad'
mg_ad = sc.read_h5ad(mg_ad_path)
mg_ad

AnnData object with n_obs × n_vars = 27177 × 10570
    obs: 'batch', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'

In [3]:
# Making a dictionary of module dataframes
module_dfs = {}

In [15]:
# Load modules dataframe: Control
module_dfs["Control"] = pd.read_csv(
    "/data/scRNA/RNA/scRNA/Hammond/preprocessing/hdWGCNA/GO/NoRP/hdWGCNA_HVG2K_normal_NoRP_modules.csv",
    index_col = "Unnamed: 0")

# Rename the modules in the modules dataframe
module_dfs["Control"]['module'] = module_dfs["Control"]['module'].str.replace("NORM-NoRP", "MG-Control")

In [6]:
# Load modules dataframe: Saline
module_dfs["Saline"] = pd.read_csv(
    "/data/scRNA/RNA/scRNA/Hammond/preprocessing/hdWGCNA/GO/NoRP/hdWGCNA_HVG2K_saline_NoRP_modules.csv",
    index_col = "Unnamed: 0")

# Rename the modules in the modules dataframe
module_dfs["Saline"]['module'] = module_dfs["Saline"]['module'].str.replace("SALINE-NoRP", "MG-Saline")

In [7]:
# Load modules dataframe: LC
module_dfs["LC"] = pd.read_csv(
    "/data/scRNA/RNA/scRNA/Hammond/preprocessing/hdWGCNA/GO/NoRP/hdWGCNA_HVG2K_injury_NoRP_modules.csv",
    index_col = "Unnamed: 0")

# Rename the modules in the modules dataframe
module_dfs["LC"]['module'] = module_dfs["LC"]['module'].str.replace("INJURY-NORP", "MG-LC")

In [16]:
from gprofiler import GProfiler

# List of all conditions
conditions = ["Control", "Saline", "LC"]

for condition in conditions:
    # List of all module names
    modules = module_dfs[condition]["module"].unique().tolist()
    modules.remove("grey")

    for module_name in modules:
        # Initialize GProfiler
        gp = GProfiler(return_dataframe=True)

        # Select genes from the specified module
        genes_of_module = module_dfs[condition][module_dfs[condition]["module"] == module_name]["gene_name"].tolist()

        # Perform GO enrichment analysis
        go_results = gp.profile(organism='mmusculus', query=genes_of_module)

        # Save the results to a CSV file
        go_results.to_csv(f"/data/scRNA/Hammond/GO/{module_name}_GO.csv")
        print(f"GO enrichment analysis results saved for module: {module_name}")

GO enrichment analysis results saved for module: MG-Control-M1
GO enrichment analysis results saved for module: MG-Control-M2
GO enrichment analysis results saved for module: MG-Control-M3
GO enrichment analysis results saved for module: MG-Control-M4
GO enrichment analysis results saved for module: MG-Saline-M1
GO enrichment analysis results saved for module: MG-Saline-M2
GO enrichment analysis results saved for module: MG-Saline-M3
GO enrichment analysis results saved for module: MG-LC-M1
GO enrichment analysis results saved for module: MG-LC-M2
GO enrichment analysis results saved for module: MG-LC-M3
